## 7-Model Stacking Pipeline

MODEL STACKING PIPELINE

- CatBoost
- LightGBM
- XGBoost
- ExtraTrees
- RandomForest

Added 
- HGB
- MLP

- OOF stacking
- Logistic meta model

- Full Optuna tuning (Cat + LGB + XGB)

## Library Import and Configuration

In [1]:
# =============================================================================
# 1. Imports & Configuration (REVISED)
# =============================================================================
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, LabelEncoder, FunctionTransformer, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

CONFIG = {
    "ADD_EXTERN_DATA": False,
    "VAL_SIZE": 0.2, 
    "N_FOLDS": 5, 
    "N_SPLITS": 5,
    "GPU_ACC": False,
    "TARGET": "Heart Disease",
    "OPTUNA_TRIALS": 11
}

NUM_COLS = ["Age", "BP", "Cholesterol", "Max HR", "ST depression"]
CAT_COLS = ["Sex", "Chest pain type", "FBS over 120", "EKG results", "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"]

print("Completed Loading Libraries and Configuration ...")


Completed Loading Libraries and Configuration ...


## Data Loading

In [2]:
# =============================================================================
# 2. Data Loading & Encoding
# =============================================================================
train_df = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test_df = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

if CONFIG["ADD_EXTERN_DATA"]:
    original_df = pd.read_csv("/kaggle/input/heart-disease-original/Heart_Disease_Prediction.csv")
    train_df = pd.concat(
        [train_df.iloc[:, 1:], original_df[train_df.columns[1:]]],
        ignore_index=True
    ).reset_index().rename(columns={"index": "id"})

# Encode target and keep the LabelEncoder instance
le = LabelEncoder()
train_df[CONFIG["TARGET"]] = le.fit_transform(train_df[CONFIG["TARGET"]])

print("Completed Loading Data ...")


Completed Loading Data ...


## Feature Engineering

In [3]:
# =============================================================================
# 3. Enhanced Feature Engineering
# =============================================================================
def add_engineered_features(df):
    df = df.copy()

    # Domain Specific Clinical Features
    df["pulse_pressure"] = df["BP"] - 80 # Assuming 80 as baseline diastolic
    df["hr_reserve"] = (220 - df["Age"]) - df["Max HR"]
    
    # Ratios
    df["chol_age_ratio"] = df["Cholesterol"] / (df["Age"] + 1)
    df["st_hr_ratio"]    = df["ST depression"] / (df["Max HR"] + 1)

    # Non-linear transforms
    df["log_age"] = np.log1p(df["Age"])
    df["sqrt_chol"] = np.sqrt(df["Cholesterol"].clip(lower=0))

    # Binary flags
    df["st_present"]  = (df["ST depression"] > 0).astype(np.uint8)

    # Interactions
    df["sex_chestpain"] = df["Sex"].astype(str) + "_" + df["Chest pain type"].astype(str)
    df["thal_vessels"]  = df["Thallium"].astype(str) + "_" + df["Number of vessels fluro"].astype(str)

    return df

train_df = add_engineered_features(train_df)
test_df  = add_engineered_features(test_df)

# Update Column Lists
NEW_NUM = ["pulse_pressure", "hr_reserve", "chol_age_ratio", "st_hr_ratio", "log_age", "sqrt_chol"]
NUM_COLS += NEW_NUM
CAT_COLS += ["sex_chestpain", "thal_vessels"]

print("Completed generating Feature Engineering ...")


Completed generating Feature Engineering ...


In [4]:
# =============================================================================
# 4. Preprocessing Pipeline (Added PCA for diversity)
# =============================================================================
X = train_df.drop(columns=[CONFIG["TARGET"], "id"])
y = train_df[CONFIG["TARGET"]]
X_test_final = test_df.drop(columns=["id"], errors='ignore')

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=CONFIG["VAL_SIZE"], stratify=y, random_state=42
)

def freq_enc(X):
    return pd.DataFrame(X).apply(lambda c: c.map(c.value_counts(normalize=True)).fillna(0))

preprocessing = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", RobustScaler())]), NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), CAT_COLS),
    ("pca", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("pca", PCA(n_components=3))]), NUM_COLS),
    ("freq", FunctionTransformer(freq_enc), CAT_COLS),
], remainder="drop")

X_train = preprocessing.fit_transform(X_train_raw)
X_val = preprocessing.transform(X_val_raw)
X_test = preprocessing.transform(X_test_final)

print("Completed Preprocessing Pipeline ...")

Completed Preprocessing Pipeline ...


## Tuning Stacking Pipeline

In [5]:
# =============================================================================
# 5. Optuna Hyperparameter Tuning
# =============================================================================
# =============================================================================
# 5. Optuna Hyperparameter Tuning (Corrected)
# =============================================================================
USE_GPU = False   

CATBOOST_DEVICE = "GPU" if USE_GPU else "CPU"
LGBM_DEVICE = "gpu" if USE_GPU else "cpu"

XGB_TREE_METHOD = "hist"  
XGB_DEVICE = "cuda" if USE_GPU else "cpu"

# FIX: X_train is already a numpy array from the ColumnTransformer. 
# We only use .values if the object is a Pandas DataFrame/Series.
X_np = X_train if isinstance(X_train, np.ndarray) else X_train.values
y_np = y_train.values if hasattr(y_train, 'values') else y_train
X_test_np = X_test if isinstance(X_test, np.ndarray) else X_test.values

def objective_cat(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_strength": trial.suggest_float("random_strength", 0.1, 1.0),
        "auto_class_weights": "Balanced",
        "eval_metric": "AUC",
        "verbose": False,
        "allow_writing_files": False,
        "task_type": CATBOOST_DEVICE,
        "random_state": 42
    }

    # Use the CONFIG dictionary defined earlier for N_SPLITS
    cv = StratifiedKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=42)
    scores = []

    for tr, ev in cv.split(X_np, y_np):
        model = CatBoostClassifier(**params)

        model.fit(
            X_np[tr], y_np[tr],
            # FIX: eval_set should be a list of tuples [(X, y)]
            eval_set=[(X_np[ev], y_np[ev])], 
            early_stopping_rounds=100,
            verbose=False
        )

        preds = model.predict_proba(X_np[ev])[:, 1]
        scores.append(roc_auc_score(y_np[ev], preds))

    return np.mean(scores)


print("Preprocessing Catboost Completed Successfully.")


Preprocessing Catboost Completed Successfully.


In [6]:
# =============================================================================
# Tuning LGBM Parameters
# =============================================================================
def objective_lgbm(trial):

    params = {
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "device": LGBM_DEVICE,
        "random_state": 42
    }

    cv = StratifiedKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=42)
    scores = []

    for tr, ev in cv.split(X_np, y_np):

        model = LGBMClassifier(**params)

        model.fit(
            X_np[tr], y_np[tr],
            eval_set=[(X_np[ev], y_np[ev])],
            eval_metric="auc",
            callbacks=[early_stopping(100, verbose=False)]
        )

        preds = model.predict_proba(X_np[ev])[:, 1]
        scores.append(roc_auc_score(y_np[ev], preds))

    return np.mean(scores)

print("Preprocessing LGBM Classification Completed Successfully.")

Preprocessing LGBM Classification Completed Successfully.


In [7]:
# =============================================================================
# Tuning XGBoost Parameters
# =============================================================================
def objective_xgb(trial):

    params = {
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "eval_metric": "auc",
        "tree_method": XGB_TREE_METHOD,
        "device": XGB_DEVICE,
        "random_state": 42
    }

    cv = StratifiedKFold(
        n_splits=CONFIG["N_SPLITS"],
        shuffle=True,
        random_state=42
    )

    scores = []

    for tr, ev in cv.split(X_np, y_np):

        model = XGBClassifier(**params)

        model.fit(
            X_np[tr],
            y_np[tr],
            eval_set=[(X_np[ev], y_np[ev])],
            verbose=False
        )

        preds = model.predict_proba(X_np[ev])[:, 1]
        scores.append(roc_auc_score(y_np[ev], preds))

    return np.mean(scores)

print("Preprocessing XGBboost Completed Successfully.")

Preprocessing XGBboost Completed Successfully.


In [8]:
# =============================================================================
# Tuning MLP Parameters
# =============================================================================
def objective_mlp(trial):
    params = {
        "hidden_layer_sizes": trial.suggest_categorical("hidden_layer_sizes", [(64,), (64, 32), (128, 64)]),
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        "max_iter": 500,
        "random_state": 42
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, ev in cv.split(X_train, y_train):
        model = MLPClassifier(**params)
        model.fit(X_train[tr], y_train.iloc[tr])
        scores.append(roc_auc_score(y_train.iloc[ev], model.predict_proba(X_train[ev])[:, 1]))
    return np.mean(scores)

print("Preprocessing MLP Completed Successfully.")

Preprocessing MLP Completed Successfully.


In [9]:
# =============================================================================
# Tuning HGB Parameters
# =============================================================================
def objective_hgb(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 50),
        "random_state": 42
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, ev in cv.split(X_train, y_train):
        model = HistGradientBoostingClassifier(**params)
        model.fit(X_train[tr], y_train.iloc[tr])
        scores.append(roc_auc_score(y_train.iloc[ev], model.predict_proba(X_train[ev])[:, 1]))
    return np.mean(scores)

print("Preprocessing HGB Completed Successfully.")

Preprocessing HGB Completed Successfully.


In [10]:

def logging_callback(study, trial):
    print(
        f"Trial {trial.number:03d} | "
        f"Value: {trial.value:.5f} | "
        f"Best: {study.best_value:.5f}",
        end="\r"
    )


## Run Studies

In [11]:
# =============================================================================
# 6. Run Studies
# =============================================================================
print("\nStarting Model Studies")
print("\nStarting CatBoost tuning...")

study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=CONFIG["OPTUNA_TRIALS"], callbacks=[logging_callback])
print(f"\nCatBoost Best AUC: {study_cat.best_value:.6f}")


[I 2026-02-22 02:00:05,155] A new study created in memory with name: no-name-f8633de7-b31b-4946-83cd-45bc3bd6aa50



Starting Model Studies

Starting CatBoost tuning...


[I 2026-02-22 02:08:57,089] Trial 0 finished with value: 0.9541406516526596 and parameters: {'learning_rate': 0.0069199876926961555, 'depth': 9, 'l2_leaf_reg': 7.752188013119593, 'random_strength': 0.6162568117291134}. Best is trial 0 with value: 0.9541406516526596.


[I 2026-02-22 02:14:29,177] Trial 1 finished with value: 0.9550081675373889 and parameters: {'learning_rate': 0.0577780597863119, 'depth': 7, 'l2_leaf_reg': 6.272201042912026, 'random_strength': 0.880842533772952}. Best is trial 1 with value: 0.9550081675373889.


[I 2026-02-22 02:29:07,462] Trial 2 finished with value: 0.9540659455220943 and parameters: {'learning_rate': 0.0061866345079617255, 'depth': 10, 'l2_leaf_reg': 5.4113117669343325, 'random_strength': 0.687766565530115}. Best is trial 1 with value: 0.9550081675373889.


...

[I 2026-02-22 03:07:12,310] Trial 7 finished with value: 0.9550108702382275 and parameters: {'learning_rate': 0.04602316792215497, 'depth': 7, 'l2_leaf_reg': 7.019899421847568, 'random_strength': 0.543255077282806}. Best is trial 4 with value: 0.9551024541091329.


[I 2026-02-22 03:21:35,155] Trial 8 finished with value: 0.9545668516442767 and parameters: {'learning_rate': 0.015236073192988978, 'depth': 10, 'l2_leaf_reg': 1.0174933314900017, 'random_strength': 0.4834541326420855}. Best is trial 4 with value: 0.9551024541091329.


[I 2026-02-22 03:27:30,963] Trial 9 finished with value: 0.954841831291869 and parameters: {'learning_rate': 0.02546685235157449, 'depth': 6, 'l2_leaf_reg': 6.385467623452645, 'random_strength': 0.8960708938257957}. Best is trial 4 with value: 0.9551024541091329.


[I 2026-02-22 03:32:29,466] Trial 10 finished with value: 0.9551375862862829 and parameters: {'learning_rate': 0.09024386422257827, 'depth': 4, 'l2_leaf_reg': 9.852038139414029, 'random_strength': 0.17730357365353122}. Best is trial 10 with value: 0.9551375862862829.


Trial 010 | Value: 0.95514 | Best: 0.95514
CatBoost Best AUC: 0.955138


In [12]:
print("\nStarting LightGBM tuning...")
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=CONFIG["OPTUNA_TRIALS"], callbacks=[logging_callback])
print(f"\nLightGBM Best AUC: {study_lgbm.best_value:.6f}")


[I 2026-02-22 03:32:29,492] A new study created in memory with name: no-name-29727a2f-18a0-40e0-bef8-cf556da6ebd5



Starting LightGBM tuning...


[I 2026-02-22 03:34:48,591] Trial 0 finished with value: 0.9550426450851284 and parameters: {'learning_rate': 0.06430239689528783, 'num_leaves': 43, 'max_depth': 4, 'subsample': 0.6243167527256713, 'colsample_bytree': 0.8614138358394194, 'lambda_l1': 0.234930466504192, 'lambda_l2': 2.661544835438791}. Best is trial 0 with value: 0.9550426450851284.


[I 2026-02-22 03:40:33,611] Trial 1 finished with value: 0.9543607181928738 and parameters: {'learning_rate': 0.009531122802771227, 'num_leaves': 105, 'max_depth': 7, 'subsample': 0.8108770537936147, 'colsample_bytree': 0.9089976038105881, 'lambda_l1': 0.026490623456364028, 'lambda_l2': 9.14670014805462e-06}. Best is trial 0 with value: 0.9550426450851284.


[I 2026-02-22 03:44:49,454] Trial 2 finished with value: 0.954335237106565 and parameters: {'learning_rate': 0.010418463619774209, 'num_leaves': 84, 'max_depth': 5, 'subsample': 0.6397018693939195, 'colsample_bytree': 0.7120294714365061, 'lambda_l1': 3.0750586805391203e-07, 'lambda_l2': 0.016473803870053292}. Best is trial 0 with value: 0.9550426450851284.


...

[I 2026-02-22 04:00:20,554] Trial 7 finished with value: 0.9548316203906577 and parameters: {'learning_rate': 0.03262252241251898, 'num_leaves': 75, 'max_depth': 6, 'subsample': 0.8253843633881224, 'colsample_bytree': 0.8087350703880698, 'lambda_l1': 0.0008429588512458144, 'lambda_l2': 9.652820067091697e-06}. Best is trial 0 with value: 0.9550426450851284.


[I 2026-02-22 04:03:36,181] Trial 8 finished with value: 0.954510814509599 and parameters: {'learning_rate': 0.029536475671551335, 'num_leaves': 158, 'max_depth': 11, 'subsample': 0.8002902261648084, 'colsample_bytree': 0.7836840710901731, 'lambda_l1': 2.6746340121596104, 'lambda_l2': 2.440460810505209e-08}. Best is trial 0 with value: 0.9550426450851284.


[I 2026-02-22 04:06:58,799] Trial 9 finished with value: 0.9545220763196142 and parameters: {'learning_rate': 0.02712545749516095, 'num_leaves': 155, 'max_depth': 11, 'subsample': 0.7512015966668322, 'colsample_bytree': 0.6867678344215196, 'lambda_l1': 4.359424046741199e-08, 'lambda_l2': 0.003049519601077142}. Best is trial 0 with value: 0.9550426450851284.


[I 2026-02-22 04:09:23,097] Trial 10 finished with value: 0.9550894236408723 and parameters: {'learning_rate': 0.0955943534889065, 'num_leaves': 31, 'max_depth': 3, 'subsample': 0.6323683958881692, 'colsample_bytree': 0.8836252215546728, 'lambda_l1': 3.118956104638532, 'lambda_l2': 7.714704331739971}. Best is trial 10 with value: 0.9550894236408723.


Trial 010 | Value: 0.95509 | Best: 0.95509
LightGBM Best AUC: 0.955089


In [13]:
print("\nStarting XGBoost tuning...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb,n_trials=CONFIG["OPTUNA_TRIALS"], callbacks=[logging_callback])
print(f"\nXGBoost Best AUC: {study_xgb.best_value:.6f}")



[I 2026-02-22 04:09:23,126] A new study created in memory with name: no-name-e125b31b-dc24-4daf-b761-d7b6279d6390



Starting XGBoost tuning...


[I 2026-02-22 04:12:03,648] Trial 0 finished with value: 0.9548722615406622 and parameters: {'learning_rate': 0.025221397750733707, 'max_depth': 4, 'subsample': 0.5324751594681771, 'colsample_bytree': 0.8136933360350127}. Best is trial 0 with value: 0.9548722615406622.


[I 2026-02-22 04:16:29,839] Trial 1 finished with value: 0.9543115459706751 and parameters: {'learning_rate': 0.017079828465823437, 'max_depth': 9, 'subsample': 0.8290957622011956, 'colsample_bytree': 0.7685457983987944}. Best is trial 0 with value: 0.9548722615406622.


[I 2026-02-22 04:21:28,702] Trial 2 finished with value: 0.9533590888362052 and parameters: {'learning_rate': 0.023324694168408155, 'max_depth': 10, 'subsample': 0.6255012247448783, 'colsample_bytree': 0.9632433639378755}. Best is trial 0 with value: 0.9548722615406622.


...

[I 2026-02-22 04:34:55,216] Trial 7 finished with value: 0.9548841406862417 and parameters: {'learning_rate': 0.03453415686359548, 'max_depth': 3, 'subsample': 0.9604196166126523, 'colsample_bytree': 0.9107957575472536}. Best is trial 3 with value: 0.9550223542333361.


[I 2026-02-22 04:38:41,759] Trial 8 finished with value: 0.9539932394977317 and parameters: {'learning_rate': 0.031633735106219266, 'max_depth': 8, 'subsample': 0.5455351661009156, 'colsample_bytree': 0.7480865832908323}. Best is trial 3 with value: 0.9550223542333361.


[I 2026-02-22 04:42:23,145] Trial 9 finished with value: 0.952360138614635 and parameters: {'learning_rate': 0.0642431362617678, 'max_depth': 9, 'subsample': 0.9501202093733114, 'colsample_bytree': 0.620801349690784}. Best is trial 3 with value: 0.9550223542333361.


[I 2026-02-22 04:45:24,233] Trial 10 finished with value: 0.953326052779208 and parameters: {'learning_rate': 0.09836882153714027, 'max_depth': 6, 'subsample': 0.6807925719742941, 'colsample_bytree': 0.6724623341485655}. Best is trial 3 with value: 0.9550223542333361.


Trial 010 | Value: 0.95333 | Best: 0.95502
XGBoost Best AUC: 0.955022


In [14]:
print("\nStarting MLP tuning...")
study_mlp = optuna.create_study(direction="maximize")
study_mlp.optimize(objective_mlp, n_trials=CONFIG["OPTUNA_TRIALS"], callbacks=[logging_callback])
print(f"\nMLP Best AUC: {study_mlp.best_value:.6f}")



[I 2026-02-22 04:45:24,266] A new study created in memory with name: no-name-158b9110-7196-439b-86c4-67297812be17



Starting MLP tuning...


[I 2026-02-22 04:53:24,298] Trial 0 finished with value: 0.9529200374944239 and parameters: {'hidden_layer_sizes': (64, 32), 'alpha': 0.0019497088236233476, 'learning_rate_init': 0.003318428519958106}. Best is trial 0 with value: 0.9529200374944239.


[I 2026-02-22 05:01:16,718] Trial 1 finished with value: 0.9527600261287761 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 0.06351497779701858, 'learning_rate_init': 0.0021123146073393677}. Best is trial 0 with value: 0.9529200374944239.


[I 2026-02-22 05:19:25,162] Trial 2 finished with value: 0.9517921484703811 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 1.956431836753503e-05, 'learning_rate_init': 0.00022633946831417858}. Best is trial 0 with value: 0.9529200374944239.


...

[I 2026-02-22 06:22:19,863] Trial 7 finished with value: 0.9525542332302712 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 3.969838088839577e-05, 'learning_rate_init': 0.00418605230080224}. Best is trial 5 with value: 0.9529307950501678.


[I 2026-02-22 06:27:54,238] Trial 8 finished with value: 0.9527299409529778 and parameters: {'hidden_layer_sizes': (64,), 'alpha': 0.0001798861683720175, 'learning_rate_init': 0.007081889525518603}. Best is trial 5 with value: 0.9529307950501678.


[I 2026-02-22 06:41:25,393] Trial 9 finished with value: 0.9529333345849219 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 0.011952292780193273, 'learning_rate_init': 0.0012069574681286253}. Best is trial 9 with value: 0.9529333345849219.


[I 2026-02-22 06:53:14,115] Trial 10 finished with value: 0.9529143854144224 and parameters: {'hidden_layer_sizes': (64, 32), 'alpha': 0.08416163901211127, 'learning_rate_init': 0.0004086975595687835}. Best is trial 9 with value: 0.9529333345849219.


Trial 010 | Value: 0.95291 | Best: 0.95293
MLP Best AUC: 0.952933


In [15]:
print("\nStarting HGB tuning...")
study_hgb = optuna.create_study(direction="maximize")
study_hgb.optimize(objective_hgb, n_trials=CONFIG["OPTUNA_TRIALS"], callbacks=[logging_callback])
print(f"\nXGBoost Best AUC: {study_hgb.best_value:.6f}")


[I 2026-02-22 06:53:14,175] A new study created in memory with name: no-name-dee6637a-390a-46a4-9cc1-3baee2b43585



Starting HGB tuning...


[I 2026-02-22 06:53:34,773] Trial 0 finished with value: 0.9478410779800163 and parameters: {'learning_rate': 0.010407057606115825, 'max_leaf_nodes': 45, 'max_depth': 5, 'min_samples_leaf': 50}. Best is trial 0 with value: 0.9478410779800163.


[I 2026-02-22 06:53:55,927] Trial 1 finished with value: 0.9543841024155612 and parameters: {'learning_rate': 0.1314267732258011, 'max_leaf_nodes': 53, 'max_depth': 6, 'min_samples_leaf': 39}. Best is trial 1 with value: 0.9543841024155612.


[I 2026-02-22 06:54:18,668] Trial 2 finished with value: 0.9543454370045156 and parameters: {'learning_rate': 0.10700271184079718, 'max_leaf_nodes': 45, 'max_depth': 7, 'min_samples_leaf': 48}. Best is trial 1 with value: 0.9543841024155612.


...

[I 2026-02-22 06:56:12,174] Trial 7 finished with value: 0.948142927015013 and parameters: {'learning_rate': 0.01897044109895195, 'max_leaf_nodes': 15, 'max_depth': 5, 'min_samples_leaf': 36}. Best is trial 3 with value: 0.9543943747985454.


[I 2026-02-22 06:56:37,272] Trial 8 finished with value: 0.950634077455642 and parameters: {'learning_rate': 0.019262225478879277, 'max_leaf_nodes': 38, 'max_depth': 8, 'min_samples_leaf': 28}. Best is trial 3 with value: 0.9543943747985454.


[I 2026-02-22 06:56:55,304] Trial 9 finished with value: 0.9475011595365129 and parameters: {'learning_rate': 0.01692236582847809, 'max_leaf_nodes': 49, 'max_depth': 4, 'min_samples_leaf': 38}. Best is trial 3 with value: 0.9543943747985454.


[I 2026-02-22 06:57:10,119] Trial 10 finished with value: 0.95415285772798 and parameters: {'learning_rate': 0.19498740639784415, 'max_leaf_nodes': 62, 'max_depth': 3, 'min_samples_leaf': 15}. Best is trial 3 with value: 0.9543943747985454.


Trial 010 | Value: 0.95415 | Best: 0.95439
XGBoost Best AUC: 0.954394


In [16]:
best_cat = study_cat.best_params
best_lgbm = study_lgbm.best_params
best_xgb = study_xgb.best_params
best_mlp = study_mlp.best_params
best_hgb = study_hgb.best_params
print("Optuna Parametrization Completed Successfully.")

Optuna Parametrization Completed Successfully.


## Model Stacking

In [17]:
# =============================================================================
# 7. Model Stacking Routine
# =============================================================================
print("\nStarting Model Stacking...")

import time

n_models = 7
oof = np.zeros((len(X_train), n_models))
test_pred = np.zeros((len(X_test), n_models))

# Use N_SPLITS from config
cv = StratifiedKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=42)

print(f"\n{'='*50}")
print(f"🚀 STARTING STACKING ROUTINE: {n_models} MODELS | {CONFIG['N_SPLITS']} FOLDS")
print(f"{'='*50}\n")

total_start_time = time.time()

for fold, (tr, ev) in enumerate(cv.split(X_train, y_train)):
    fold_start_time = time.time()
    print(f"--- 📂 FOLD {fold + 1} / {CONFIG['N_SPLITS']} ---")
    
    # Local slice for current fold
    X_tr, X_ev = X_train[tr], X_train[ev]
    y_tr, y_ev = y_train.iloc[tr], y_train.iloc[ev]
    
    # Model Dictionary using optimized parameters from Optuna
    models = {
        "Cat": CatBoostClassifier(**study_cat.best_params, verbose=False, allow_writing_files=False),
        "LGB": LGBMClassifier(**study_lgbm.best_params, verbose=-1),
        "XGB": XGBClassifier(**study_xgb.best_params),
        "HGB": HistGradientBoostingClassifier(**study_hgb.best_params),
        "MLP": MLPClassifier(**study_mlp.best_params),
        "ET":  ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1),
        "RF":  RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    }
    
    for i, (name, model) in enumerate(models.items()):
        model_start_time = time.time()
        
        # 1. Fit
        model.fit(X_tr, y_tr)
        
        # 2. Predict Out-of-Fold (OOF)
        fold_preds = model.predict_proba(X_ev)[:, 1]
        oof[ev, i] = fold_preds
        
        # 3. Predict Test Set (Average over folds)
        test_pred[:, i] += model.predict_proba(X_test)[:, 1] / CONFIG["N_SPLITS"]
        
        # 4. Calculate Fold Performance
        fold_auc = roc_auc_score(y_ev, fold_preds)
        model_duration = time.time() - model_start_time
        
        # Log individual model results
        print(f"   [Model: {name:4}] | AUC: {fold_auc:.5f} | Time: {model_duration:5.2f}s")
    
    fold_duration = time.time() - fold_start_time
    print(f"\n✔️ Fold {fold + 1} completed in {fold_duration:.2f}s\n")

total_duration = time.time() - total_start_time
print(f"{'='*50}")
print(f"✅ STACKING COMPLETED SUCCESSFULLY in {total_duration/60:.2f} minutes.")
print(f"{'='*50}")




Starting Model Stacking...

🚀 STARTING STACKING ROUTINE: 7 MODELS | 5 FOLDS

--- 📂 FOLD 1 / 5 ---
   [Model: Cat ] | AUC: 0.95475 | Time: 52.62s
   [Model: LGB ] | AUC: 0.95310 | Time:  4.28s
   [Model: XGB ] | AUC: 0.95310 | Time:  2.80s
   [Model: HGB ] | AUC: 0.95421 | Time:  9.08s
   [Model: MLP ] | AUC: 0.95249 | Time: 549.85s
   [Model: ET  ] | AUC: 0.94472 | Time: 169.99s
   [Model: RF  ] | AUC: 0.94805 | Time: 269.82s

✔️ Fold 1 completed in 1058.50s

--- 📂 FOLD 2 / 5 ---
   [Model: Cat ] | AUC: 0.95491 | Time: 52.48s
   [Model: LGB ] | AUC: 0.95330 | Time:  4.26s
   [Model: XGB ] | AUC: 0.95325 | Time:  2.64s
   [Model: HGB ] | AUC: 0.95431 | Time:  7.27s
   [Model: MLP ] | AUC: 0.95253 | Time: 352.47s
   [Model: ET  ] | AUC: 0.94433 | Time: 197.24s
   [Model: RF  ] | AUC: 0.94751 | Time: 269.38s

✔️ Fold 2 completed in 885.91s

--- 📂 FOLD 3 / 5 ---
   [Model: Cat ] | AUC: 0.95530 | Time: 51.87s
   [Model: LGB ] | AUC: 0.95354 | Time:  4.44s
   [Model: XGB ] | AUC: 0.95352 | 

## Generate Meta Model

In [18]:
# =============================================================================
# 8. Generate Meta Model
# =============================================================================
print("\nStarting Meta Modeling...")

from sklearn.linear_model import LogisticRegressionCV

meta_model = LogisticRegressionCV(scoring='roc_auc', random_state=42)
meta_model.fit(oof, y_train)

final_preds = meta_model.predict_proba(test_pred)[:, 1] # Use predict_proba for Logistic




Starting Meta Modeling...


## Submission

In [19]:
# =============================================================================
# 9. Submission
# =============================================================================
submission = pd.DataFrame({"id": test_df["id"], CONFIG["TARGET"]: final_preds})
submission.to_csv("submission.csv", index=False)
print(f"Final Stacking ROC-AUC: {roc_auc_score(y_train, meta_model.predict(oof)):.6f}")

print("\nSubmission Created Successfully.")

submission.head(20)

Final Stacking ROC-AUC: 0.886108

Submission Created Successfully.


,id,Heart Disease
0,630000,0.947707
1,630001,0.038853
2,630002,0.959995
3,630003,0.038153
4,630004,0.125215
5,630005,0.960281
6,630006,0.038576
7,630007,0.726219
8,630008,0.961778
9,630009,0.040072
